# 24 — Automatic Email Triage
> Each incoming email is read by a language model, which assigns a priority level, a category, a one-sentence summary, and a concrete next action — guaranteed in structured form, ready to route.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q openai pydantic python-dotenv
import os
os.environ['OPENROUTER_API_KEY'] = 'your-openrouter-key'  # replace

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from openai import OpenAI


class EmailTriage(BaseModel):
    urgency: Literal["high", "medium", "low"] = Field(
        description="How urgently this email needs a response."
    )
    category: Literal["billing", "technical", "general", "spam"] = Field(
        description="The primary topic category of the email."
    )
    summary: str = Field(
        description="One-sentence plain-English summary of what the email is about."
    )
    recommended_action: str = Field(
        description="Concrete next step the recipient should take (e.g. 'Escalate to billing team')."
    )


def create_client() -> OpenAI:
    return OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )


def classify(email_text: str, model: str = "openai/gpt-4o-mini") -> EmailTriage:
    client = create_client()
    completion = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an email triage assistant. "
                    "Classify the email and recommend an action."
                ),
            },
            {"role": "user", "content": email_text},
        ],
        response_format=EmailTriage,
    )
    return completion.choices[0].message.parsed

## The scenario

A customer success manager starts Monday with five unread emails in a shared inbox. One is a refund dispute that has been waiting since Friday, one is a broken API report from a paying enterprise client, and the rest are a mix of product feedback, a phishing attempt, and a routine renewal reminder.

The agent reads each email and returns exactly what the team needs to act: which ones are on fire, what bucket each belongs to, and the one thing to do next.

In [ ]:
EMAILS = [
    (
        "Refund dispute",
        """Subject: Re: Re: Re: Refund for order #9912 — still waiting
From: priya.mehta@outlook.com

I submitted my refund request three weeks ago and I still haven't received
my money back. I was charged $318 for a subscription I cancelled before
the renewal date. I've emailed twice already. If this isn't resolved by
Friday I'm filing a chargeback with my bank.""",
    ),
    (
        "API broken",
        """Subject: CRITICAL: /v2/export endpoint returning 500 since 09:14 UTC
From: devops@enterprise-client.io

Our nightly data export pipeline has been failing since 09:14 UTC this morning.
Every call to /v2/export returns HTTP 500 with no body. We're on the
Enterprise plan and this is blocking our end-of-month reporting.
We need a status update within the hour.""",
    ),
    (
        "Product feedback",
        """Subject: Small suggestion — bulk-select on the reports page
From: tom.weller@startup.co

Quick one: would love a 'select all' checkbox on the reports page so I can
export or delete multiple reports at once. Right now I have to click each
one individually which gets tedious when I have 30+ reports. Love the product
otherwise, keep it up!""",
    ),
    (
        "Phishing attempt",
        """Subject: Your account requires immediate verification
From: noreply@secure-account-verify.net

Your account has been flagged for suspicious activity. Click the link below
within 24 hours to verify your identity or your account will be permanently
disabled. Verify now: http://secure-account-verify.net/login?token=xg99z""",
    ),
    (
        "Renewal reminder",
        """Subject: Your annual plan renews in 14 days
From: billing@saas-platform.com

Just a heads-up that your Annual Pro plan will automatically renew on
July 2nd for $840. No action is needed if you'd like to continue.
If you'd like to change your plan or update your payment method, log in
to your account settings before the renewal date.""",
    ),
]

# Run triage
results = []
for label, email in EMAILS:
    result = classify(email)
    results.append((label, result))

# Print results
URGENCY_ICON = {"high": "[HIGH]", "medium": "[MED] ", "low": "[LOW] "}

print(f"{'EMAIL':<22} {'PRIORITY':<8}  {'CATEGORY':<12}  SUMMARY & ACTION")
print("-" * 95)
for label, r in results:
    icon = URGENCY_ICON[r.urgency]
    print(f"{label:<22} {icon}  {r.category:<12}  {r.summary}")
    print(f"{'':>34} Next: {r.recommended_action}")
    print()

## Use your own data

Replace the `EMAILS` list above with:
- Your raw email text (paste the full body, including Subject and From lines)
- As many emails as you like — one tuple per email

The agent will return a priority level, category, one-line summary, and a concrete next action for every email you give it.